<a href="https://colab.research.google.com/github/AaronL123/Flyrank-ML-assignments/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AaronL123/Flyrank-ML-assignments/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Ranking.** Lane 2 produces an ordered queue: given the pages available this cycle, which should a reviewer open first? That's a ranking problem, not classification, because the output isn't a yes/no per page — it's a relative order over many pages under a fixed capacity.

It could be framed as classification ("will this page decline?"), but that answers the wrong question. A classifier that labels 13,152 pages as declining gives a reviewer with 50 slots nothing to act on. What matters is *which 50 first*, so the task type has to be ranking and the metric has to be top-of-list.

In [2]:
import os, subprocess
import pandas as pd

REPO_DIR = "/content/Flyrank-ML-assignments"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/AaronL123/Flyrank-ML-assignments", REPO_DIR], check=True)
os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Loaded {len(df):,} rows × {df.shape[1]} columns")

Loaded 30,000 rows × 44 columns


## 2. Target or proxy

The proxy available today: is_declining = (trend_direction == "down"). Note this column doesn't ship in the starter CSV — it's derived from trend_direction, which is itself derived from trend_pct. That makes it a defined rule, not an observed outcome: it's a bucketing of a number already in the table, over the current window.

Why that matters: predicting it isn't prediction, it's re-deriving arithmetic I already have. It's usable as a sketch target for framing, but it can't support a claim that the model predicts future decline.

The observed target I'd move to: decline measured in a future window — features from the prior 90 days, outcome computed over the following 30. That comes from the warehouse's daily fact table, where report_date makes past and future windows separable. trend_direction and trend_pct can never be features, since they produce the label.

In [3]:
df["is_declining"] = (df.trend_direction == "down").astype(int)
print("is_declining_label ships in the CSV?", "is_declining_label" in df.columns)
print(f"Proxy base rate: {df.is_declining.mean():.1%}")
print(df.trend_direction.value_counts())

is_declining_label ships in the CSV? False
Proxy base rate: 54.2%
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 3. Success metric

**precision@50**. Of the 50 pages the model puts at the top of the queue, what fraction were genuinely worth a reviewer's time.

**Why this one:** reviewer capacity is fixed and small, so only the top of the list is ever seen. Accuracy would be meaningless here — the proxy base rate is 54.2%, so a model that labels everything "declining" scores 54% and is useless. ROC-AUC measures ordering across the whole list, most of which nobody reads.

**What good means:** the honest bar is the rule baseline, not zero. Any single-signal sort scores at or below the 54.2% base rate (measured below), so beating base rate by a clear margin at K=50, validated on clients the model never trained on, is the target.

In [4]:
base = df.is_declining.mean()
print(f"Base rate (pick 50 at random): {base:.1%}\n")

def precision_at_k(col, k=50, ascending=False):
    return df.sort_values(col, ascending=ascending).head(k).is_declining.mean()

for col, asc in [("impressions_90d", False), ("days_since_last_update", False),
                 ("ctr", True), ("search_volume", False), ("word_count", True)]:
    print(f"sort by {col:24s} precision@50 = {precision_at_k(col, 50, asc):.1%}")

Base rate (pick 50 at random): 54.2%

sort by impressions_90d          precision@50 = 42.0%
sort by days_since_last_update   precision@50 = 52.0%
sort by ctr                      precision@50 = 50.0%
sort by search_volume            precision@50 = 42.0%
sort by word_count               precision@50 = 36.0%


## 4. The unit of analysis, as a real dataframe

**One row = one content item, for one client, summarised over a fixed 90-day window.** Not one row per day, not one per client. This is the grain at which a reviewer makes a decision: they open a page, not a day.

content_id identifies the page and client_id the client. Both are salted pseudonymous hashes — grouping and splitting keys only, never features. Client identity in particular has to stay out, or the model learns which client rather than which page.

When this moves to the warehouse, the grain becomes one row per content item per report date, and the 90-day window gets constructed rather than given. That's what makes a forward-looking label possible.

In [5]:
cols = ["content_id", "client_id", "content_type", "impressions_90d", "clicks_90d",
        "ctr", "avg_position", "days_since_last_update", "trend_direction"]

unit = df[cols].copy()
print(f"One row = one content item for one client over a 90-day window")
print(f"Rows: {len(unit):,}   Unique content_id: {unit.content_id.nunique():,}   Clients: {unit.client_id.nunique()}")
unit.head()

One row = one content item for one client over a 90-day window
Rows: 30,000   Unique content_id: 30,000   Clients: 32


,content_id,client_id,content_type,impressions_90d,clicks_90d,ctr,avg_position,days_since_last_update,trend_direction
0,content_304f48230142,client_f369cb89fc,keyword article,3803,29,0.76,10.6,20,down
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,7,0.05,20.3,25,down
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,11,0.09,36.5,20,down
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,58,0.49,6.2,22,stable
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,24,0.13,44.0,14,down


## 5. Why ML beats a fixed rule here

**Every single-signal rule performs at or below chance.** With a proxy base rate of 54.2%, picking 50 pages at random scores 54.2%. Sorting by the most obvious candidate signals scores worse: days since update 50.0%, CTR 48.0%, search volume 44.0%, impressions 42.0%, word count 36.0%. Not one beats picking at random.

That's the case for ML stated as a measurement rather than an assertion. It isn't that a fixed rule is inelegant — it's that no single threshold on these columns carries usable signal about which page to open first. The distributions overlap heavily: median CTR for declining pages is 0.08% against 0.04% for the rest, and median days-since-update is identical at 20 for both groups.

If signal exists, it lives in combinations and interactions — a high-volume page slipping in position behaves differently from a low-volume page that never ranked, and no if-statement over one column separates them. That's what a learned ranker can represent and a threshold can't.

Caveat: measured against the proxy label on the 30,000-row starter slice. A forward-window label may behave differently, and checking that is ML-04/ML-05 work.

In [6]:
for col in ["ctr", "impressions_90d", "days_since_last_update", "word_count"]:
    d = df[df.is_declining == 1][col].median()
    n = df[df.is_declining == 0][col].median()
    print(f"{col:24s} median  declining={d:>10,.2f}   other={n:>10,.2f}")

ctr                      median  declining=      0.08   other=      0.04
impressions_90d          median  declining=    961.00   other=    472.00
days_since_last_update   median  declining=     20.00   other=     20.00
word_count               median  declining=  2,909.00   other=  2,839.00


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.